# MCP Gateway Test Client

Use this notebook to connect to the MCP gateway running in Docker, inspect its tools/resources, and call a tool.

Before running the cells:

1. Start Postgres + the gateway (rebuild so policy code is in the image):
   `docker compose up -d --build postgres gateway`
2. Confirm healthy: `docker compose ps` and `curl -s http://localhost:8000/health`
3. Select the project's `.venv` Python kernel in Cursor/Jupyter.

The gateway requires a Bearer API key. Local seed key (dev only):
`aisk_dev_local_00000000000000000001`.

This notebook opens **one MCP session** (`await mcp.connect()`), reuses it for every
tool call, then closes it with `await mcp.close()` (or re-run connect to replace it).

After connect you can fetch the per-session `data_key` with only the API key:
`GET /sessions/data-key` + `Authorization: Bearer <MCP_API_KEY>`
(`await mcp.get_data_key()`).

Every allowed tool call prints a `Gateway report:` block showing what the gateway
did to the query and the result (`mcp.report(result)` returns it as a dict).

The notebook only connects to `MCP_URL`; it never starts or stops the gateway.
Policy YAML under `policies/` is bind-mounted into the container — edit and restart
the gateway to pick up changes (`docker compose restart gateway`).


In [117]:
from typing import Any

import httpx
from fastmcp import Client
from fastmcp.client.client import CallToolResult
from fastmcp.exceptions import ToolError

GATEWAY_BASE_URL = "http://localhost:8000"
MCP_URL = f"{GATEWAY_BASE_URL}/mcp/postgres"
# Seeded in {SAFE_DB_SCHEMA}.api_keys — local/dev only.
MCP_API_KEY = "aisk_dev_local_00000000000000000001"
# Mirrors app/reporting.py so this notebook stays standalone.
REPORT_META_KEY = "aisafedb"
REPORT_SUMMARY_PREFIX = "aisafedb:"

In [118]:
class McpTestClient:
    """One long-lived MCP session shared across notebook cells."""

    def __init__(self, url: str, api_key: str, gateway_base_url: str) -> None:
        self._url: str = url
        self._api_key: str = api_key
        self._gateway_base_url: str = gateway_base_url.rstrip("/")
        self._client: Client | None = None

    @property
    def connected(self) -> bool:
        return self._client is not None

    async def connect(self) -> None:
        """Open a single MCP session (initialize + mcp-session-id)."""
        if self._client is not None:
            return
        print(f"Connecting to {self._url}")
        client: Client = Client(self._url, auth=self._api_key)
        await client.__aenter__()
        self._client = client
        print("MCP session open (reuse this client for all calls below)")

    async def close(self) -> None:
        """End the MCP session (sends DELETE / clears local client)."""
        if self._client is None:
            return
        await self._client.__aexit__(None, None, None)
        self._client = None
        print("MCP session closed")

    def _require_client(self) -> Client:
        if self._client is None:
            raise RuntimeError("Not connected — run: await mcp.connect()")
        return self._client

    def mcp_session_id(self) -> str:
        """Return the transport mcp-session-id for the open session."""
        client: Client = self._require_client()
        session_id: str | None = client.transport.get_session_id()
        if not session_id:
            raise RuntimeError("MCP session id unavailable on transport")
        return session_id

    async def get_data_key(self) -> dict[str, Any]:
        """Fetch data_key for this API key's latest open session (Bearer only)."""
        url: str = f"{self._gateway_base_url}/sessions/data-key"
        async with httpx.AsyncClient() as http:
            response: httpx.Response = await http.get(
                url,
                headers={"Authorization": f"Bearer {self._api_key}"},
            )
        response.raise_for_status()
        payload: dict[str, Any] = response.json()
        print(
            f"session_id={payload['session_id']}\n"
            f"mcp_session_id={payload['mcp_session_id']}\n"
            f"data_key={payload['data_key']}"
        )
        return payload

    async def inspect(self) -> None:
        client: Client = self._require_client()
        tools = await client.list_tools()
        print(f"\nTools ({len(tools)}):")
        for tool in tools:
            description = f" — {tool.description}" if tool.description else ""
            print(f"  {tool.name}{description}")

        resources = await client.list_resources()
        print(f"\nResources ({len(resources)}):")
        for resource in resources:
            print(f"  {resource.uri}")

    async def call_tool(
        self,
        tool_name: str,
        arguments: dict[str, Any],
    ) -> CallToolResult:
        client: Client = self._require_client()
        result = await client.call_tool(tool_name, arguments)

        print(f"Result from {tool_name} (is_error={result.is_error}):")
        for content in result.content:
            text = getattr(content, "text", None)
            if text is not None and text.startswith(REPORT_SUMMARY_PREFIX):
                continue
            print(text if text is not None else content.model_dump_json(indent=2))
        report: dict[str, Any] | None = self.report(result)
        if report is not None:
            self.print_report(report)
        return result

    @staticmethod
    def report(result: CallToolResult) -> dict[str, Any] | None:
        """Return the gateway's audit report from the result meta, if attached."""
        meta: dict[str, Any] = result.meta or {}
        report = meta.get(REPORT_META_KEY)
        return report if isinstance(report, dict) else None

    @staticmethod
    def print_report(report: dict[str, Any]) -> None:
        """Show what the gateway did to the query and the result."""
        print("\nGateway report:")
        for sql in report.get("executed_sql", []):
            print(f"  executed_sql: {sql}")
        fields = (
            "expanded_stars",
            "dropped_columns",
            "hashed_columns",
            "masked_fields",
            "removed_fields",
            "call_decision",
            "result_decision",
        )
        for field in fields:
            value = report.get(field)
            if value in (None, [], False):
                continue
            print(f"  {field}: {value}")

    async def safe_call_tool(
        self,
        tool_name: str,
        arguments: dict[str, Any],
        label: str = "",
    ) -> CallToolResult | None:
        """Call a tool but print policy/guard blocks instead of raising.

        Lets the policy-demo cells below run top to bottom even when a call is
        expected to be rejected by `SqlPolicyMiddleware` or the safety guard.
        """
        if label:
            print(f"\n=== {label} ===")
        try:
            return await self.call_tool(tool_name, arguments)
        except ToolError as err:
            print(f"BLOCKED: {err}")
            return None


In [119]:
# One MCP session for the whole notebook run.
mcp = McpTestClient(MCP_URL, MCP_API_KEY, GATEWAY_BASE_URL)
await mcp.connect()
await mcp.inspect()


Connecting to http://localhost:8000/mcp/postgres
MCP session open (reuse this client for all calls below)

Tools (1):
  query — Run a read-only SQL query

Resources (1):
  postgres://aisafe@postgres:5432/customers/schema


## Session data key

Fetches `data_key` for `MCP_API_KEY`'s latest open session:
`GET /sessions/data-key` with `Authorization: Bearer <MCP_API_KEY>`.


In [120]:
# data_key for this MCP_API_KEY's latest open session (Bearer auth only).
# Connect cell above must have opened a session first.
data_key_info = await mcp.get_data_key()
print(data_key_info["data_key"])


session_id=c0295ec3-6f62-4bec-ae0a-e8e9a0c0362d
mcp_session_id=dba7b3f2-7d00-47bb-9775-bb6e75b3a455
data_key=8aWqKZUW8ihcc4FqHwQPHam23zfNnICr6iXLD2lwB-A
8aWqKZUW8ihcc4FqHwQPHam23zfNnICr6iXLD2lwB-A


## Call a tool

The example below runs a read-only query against the Postgres MCP. Change `tool_name` and `arguments` to test another MCP server.

This is the baseline case: a narrow, allowed `SELECT`. The sections further down exercise every rule in the [`pg-readonly`](../policies/pg-readonly.yaml) SQL policy attached to this server via `policy: pg-readonly` in [`mcp-servers/postgres.yaml`](../mcp-servers/postgres.yaml).

In [121]:
result = await mcp.safe_call_tool(
    "query",
    {"sql": "SELECT COUNT(*) AS customer_count FROM customers"},
    label="Baseline: narrow read-only query (allowed)",
)


=== Baseline: narrow read-only query (allowed) ===
BLOCKED: safety guard blocked tool 'query' on server 'postgres': The query result contains sensitive customer data including personally identifiable information (PII) such as `first_name`, `last_name`, `email`, `phone`, `date_of_birth`, `ssn`, and `credit_card` fields that violate the policy of masking or blocking PII. The result shows `12` customers, which may indicate the number of records but without masking or blocking the individual data fields as required by the policy. The policy explicitly blocks `ssn` and `drop` `credit_card` fields, yet the result's


## SQL policy enforcement

`policies/pg-readonly.yaml` attaches to the `postgres` server and is enforced by `SqlPolicyMiddleware` *before* a query ever reaches the database:

- **Read-only** — only `SELECT` statements are allowed; any DML/DDL is rejected.
- **Denied keywords** — e.g. `pg_sleep`, `dblink`, `copy`.
- **Database / schema / table allow lists** — only `aisafedb.public.customers` is reachable.
- **Per-column PII actions** — `block`, `drop`, `mask`, or `allow` on `customers` columns. `drop` is rewritten out of the query, `mask` is keyed-hashed inside the query by the LLM rewriter, and `PiiMaskingMiddleware` masks result-side only if the rewrite did not run.

Every allowed call comes back with a report of what the gateway did, printed under `Gateway report:` and available as `mcp.report(result)`.

Blocked calls raise `ToolError`. `safe_call_tool` (defined above) catches it and prints the reason instead of raising, so the whole notebook can be run top to bottom.

### Read-only, denied keywords, and database/schema/table access

Each call below should be **blocked** with a clear reason.

In [122]:
# read_only: true -> any non-SELECT statement is rejected.
await mcp.safe_call_tool(
    "query",
    {"sql": "UPDATE customers SET city = 'Nowhere' WHERE id = 1"},
    label="UPDATE statement (blocked: policy is read-only)",
)

# denied_keywords: [pg_sleep, dblink, copy] -> matched regardless of context.
await mcp.safe_call_tool(
    "query",
    {"sql": "SELECT pg_sleep(1)"},
    label="pg_sleep() call (blocked: denied keyword)",
)

# access.schemas: [public] -> pg_catalog is not in the allow list.
await mcp.safe_call_tool(
    "query",
    {"sql": "SELECT * FROM pg_catalog.pg_tables LIMIT 1"},
    label="Query outside the public schema (blocked: schema not allowed)",
)

# access.tables: [public.customers] -> any other table is rejected before
# it ever reaches Postgres, so this fails even though `orders` doesn't exist.
await mcp.safe_call_tool(
    "query",
    {"sql": "SELECT * FROM public.orders LIMIT 1"},
    label="Query against a table not in the allow list (blocked: table not allowed)",
)


=== UPDATE statement (blocked: policy is read-only) ===
BLOCKED: policy blocked tool 'query' on server 'postgres': policy permits read-only SQL only

=== pg_sleep() call (blocked: denied keyword) ===
BLOCKED: policy blocked tool 'query' on server 'postgres': query contains denied keyword 'pg_sleep'

=== Query outside the public schema (blocked: schema not allowed) ===
BLOCKED: policy blocked tool 'query' on server 'postgres': schema 'pg_catalog' is not allowed

=== Query against a table not in the allow list (blocked: table not allowed) ===
BLOCKED: policy blocked tool 'query' on server 'postgres': table 'public.orders' is not allowed


### PII protection: block, drop, mask, allow

`customers` columns each declare a `pii.action` in the policy:

| Action | Columns | Behavior |
| --- | --- | --- |
| `block` | `ssn` | Query is rejected outright (including `SELECT *`) |
| `drop` | `credit_card` | Column is rewritten out of the projection before execution |
| `mask` | `first_name`, `last_name`, `email`, `phone`, `date_of_birth`, `address_line1`, `address_line2`, `ip_address` | Replaced in-query with `sha256(data_key ‖ value)`, so Postgres never returns the raw value; result-side redaction is the fallback if the rewrite fails |
| `allow` | — | Passed through untouched |

Columns with no PII rule (`id`, `city`, `state`, `country`, ...) pass through unchanged. `SELECT *` is expanded to the policy's declared column list first, so `drop` and `mask` can be applied to it.

In [123]:
# action: block -> selecting the column at all is rejected.
await mcp.safe_call_tool(
    "query",
    {"sql": "SELECT ssn FROM customers LIMIT 1"},
    label="Direct select of a blocked PII column (blocked)",
)

# SELECT * is rejected on any table with a `block` column, since it would
# expose ssn even though it is not named explicitly.
await mcp.safe_call_tool(
    "query",
    {"sql": "SELECT * FROM customers LIMIT 1"},
    label="SELECT * on a table with blocked PII columns (blocked)",
)

# action: drop -> the column is silently removed from the projection, so the
# query runs but credit_card never appears in the result or the report's SQL.
await mcp.safe_call_tool(
    "query",
    {"sql": "SELECT id, credit_card FROM customers ORDER BY id LIMIT 3"},
    label="Dropped PII column (credit_card removed from the query)",
)


=== Direct select of a blocked PII column (blocked) ===
BLOCKED: policy blocked tool 'query' on server 'postgres': column 'ssn' is blocked as PII

=== SELECT * on a table with blocked PII columns (blocked) ===
BLOCKED: policy blocked tool 'query' on server 'postgres': SELECT * may expose blocked PII columns


In [124]:
# action: mask -> allowed, but the listed columns come back redacted.
# `id` and `city` have no PII rule, so they pass through unchanged.
await mcp.safe_call_tool(
    "query",
    {
        "sql": (
            "SELECT id, first_name, email, phone, date_of_birth, city "
            "FROM customers ORDER BY id LIMIT 3"
        )
    },
    label="Masked PII columns (email/phone/etc. redacted; id/city untouched)",
)

# action: mask on ip_address -> the LLM rewrites the query so Postgres returns
# sha256(data_key || ip_address); the raw value never leaves the database.
hashed = await mcp.safe_call_tool(
    "query",
    {"sql": "SELECT id, ip_address FROM customers ORDER BY id LIMIT 3"},
    label="In-query keyed hash (ip_address)",
)


=== Masked PII columns (email/phone/etc. redacted; id/city untouched) ===
BLOCKED: safety guard blocked tool 'query' on server 'postgres': tool arguments request sensitive personal data

=== Hashed PII column (ip_address) ===
Result from query (is_error=False):
[{"id":1,"ip_address":"632d808f504ad30c502456daca382dfab46429b26cb37bbf0e23f8ba61e6102f"},{"id":2,"ip_address":"0ec91b93c9517014fe77422e801ebdf2a5f751516c6f4b218e9d4f0cff072dcc"},{"id":3,"ip_address":"ee65b70dd1677cda06bc28d893adaa93139a7832ab3000bd9f4e3a492c15e17c"}]


CallToolResult(content=[TextContent(type='text', text='[{"id":1,"ip_address":"632d808f504ad30c502456daca382dfab46429b26cb37bbf0e23f8ba61e6102f"},{"id":2,"ip_address":"0ec91b93c9517014fe77422e801ebdf2a5f751516c6f4b218e9d4f0cff072dcc"},{"id":3,"ip_address":"ee65b70dd1677cda06bc28d893adaa93139a7832ab3000bd9f4e3a492c15e17c"}]', annotations=None, meta=None)], structured_content=None, meta=None, data=None, is_error=False)

### The per-call report

Each allowed call carries a record of the gateway's work in two places:

- `result.meta["aisafedb"]` — the structured [`ToolCallReport`](../app/reporting.py): executed SQL, star expansion, dropped/hashed columns, result-side masked/removed fields, and both guard verdicts.
- the last text block of `result.content` — a one-line summary, so a calling model sees the same information.

`executed_sql` is the statement actually sent to Postgres with the session `data_key` replaced by `__DATA_KEY__`, so the report can be logged or stored without leaking the secret.

In [ ]:
import json

# Full structured report for the in-query hash call above.
report = mcp.report(hashed) if hashed is not None else None
print(json.dumps(report, indent=2) if report else "No report on this result")

# The same information as the one-line summary handed to the calling model.
if hashed is not None:
    for content in hashed.content:
        text = getattr(content, "text", "")
        if text.startswith(REPORT_SUMMARY_PREFIX):
            print(f"\nSummary block: {text}")

### Editing the policy

Change [`policies/pg-readonly.yaml`](../policies/pg-readonly.yaml) (or add a new policy file there) and restart the gateway to pick up new rules — `policies/*.yaml` is loaded once at startup, same as `mcp-servers/*.yaml`.

If you change the shape of the policy models in `app/policies/models.py`, regenerate the editor-validation schema with:

```bash
make policy-schema
```

## Close the MCP session

When finished, close the shared session so the gateway marks `closed_at`
(Streamable HTTP `DELETE`). Re-run the connect cell to open a new one.

## Docker logs

The notebook does not manage the gateway process. View its requests with:

`docker compose logs -f gateway`

In [125]:
await mcp.close()


MCP session closed
